In [ ]:
# ============================================================
# RSA-CP alignment-drift experiment with lower-bound guard
#
# Real scores and reference scores are generated from the same distribution.
# Therefore, the correct alignment is identity.
#
# We deliberately drift the alignment map away from identity:
#
#     T_delta(s) = s,                       delta = 0
#     T_delta(s) != s,                      delta > 0
#
# ============================================================

import numpy as np
import pandas as pd
import math
from scipy.stats import betabinom
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# Settings
# ============================================================

SEED = 12345

ALPHA = 0.05
BETA = 0.4

M = 15
N_REF = 1000
N_TEST = 1000
N_TRIALS = 300

ALIGNMENT_DRIFT_GRID = [
    0.0, 0.05, 0.10, 0.20, 0.30,
    0.40, 0.50, 0.75, 1.00
]

SETTINGS = ["lognormal", "student"]

OUT_DIR = Path("rsa_cp_alignment_drift_guarded")
OUT_DIR.mkdir(exist_ok=True)


# ============================================================
# Score distributions
# ============================================================

def sample_scores(rng, n, setting):
    """
    Direct score-level simulation.

    Scores represent nonconformity scores, e.g. absolute residuals.
    """

    if setting == "lognormal":
        return rng.lognormal(mean=0.0, sigma=0.6, size=n)

    if setting == "student":
        return np.abs(rng.standard_t(df=3, size=n))

    raise ValueError("setting must be 'lognormal' or 'student'")


# ============================================================
# SCP
# ============================================================

def scp_quantile(scores, alpha):
    """
    Standard split conformal threshold.

    If ceil((m+1)(1-alpha)) > m, SCP is vacuous.
    """

    scores = np.sort(np.asarray(scores, dtype=float))
    m = len(scores)

    k_scp = int(np.ceil((m + 1) * (1 - alpha)))

    if k_scp > m:
        return np.inf

    return float(scores[k_scp - 1])


def scp_include_scores(scores, real_scores, alpha):
    """
    SCP inclusion indicator for score-level simulation.
    """

    real_sorted = np.sort(np.asarray(real_scores, dtype=float))
    scores = np.asarray(scores, dtype=float)

    m = len(real_sorted)
    k_scp = int(np.ceil((m + 1) * (1 - alpha)))

    if k_scp > m:
        return np.ones_like(scores, dtype=bool)

    k = 1 + np.searchsorted(real_sorted, scores, side="right")

    return k <= k_scp


# ============================================================
# Beta-Binomial windows and Prop. 3.3 bounds
# ============================================================

def beta_binomial_windows(m, N, beta):
    """
    Compute b_-(k,beta), b_+(k,beta), for k=1,...,m+1.

    B | k ~ BetaBin(N, k, m+2-k).
    """

    bminus = np.zeros(m + 2, dtype=int)
    bplus = np.zeros(m + 2, dtype=int)

    for k in range(1, m + 2):
        bminus[k] = int(
            betabinom.ppf(beta / 2, N, k, m + 2 - k)
        )
        bplus[k] = int(
            betabinom.ppf(1 - beta / 2, N, k, m + 2 - k)
        )

    return bminus, bplus


def prop33_rank_bounds(m, N, alpha, beta):
    """
    Distribution-free finite-sample rank bounds.
    """

    bminus, bplus = beta_binomial_windows(m, N, beta)

    j_star = math.ceil((1 - alpha) * (m + N + 1))

    lower = sum(
        (k + bplus[k]) <= j_star
        for k in range(1, m + 2)
    ) / (m + 1)

    upper = sum(
        (k + bminus[k]) <= j_star
        for k in range(1, m + 2)
    ) / (m + 1)

    return lower, upper


# ============================================================
# Alignment maps
# ============================================================

def make_alignment_map(delta, real_scores=None):
    """
    Monotone alignment-drift map.

    delta = 0:
        T(s) = s.

    delta > 0:
        T(s) increasingly pushes larger scores upward.

    This creates controlled misalignment while real/reference score
    distributions remain the same.
    """

    if real_scores is None:
        scale = 1.0
    else:
        scale = np.subtract(*np.percentile(real_scores, [75, 25]))

        if not np.isfinite(scale) or scale < 1e-8:
            scale = np.std(real_scores)

        if not np.isfinite(scale) or scale < 1e-8:
            scale = 1.0

    def T(values):
        values = np.asarray(values, dtype=float)

        if delta == 0:
            return values

        # Monotone upper-tail distortion.
        z = values + delta * (values ** 2) / (scale + np.abs(values) + 1e-8)

        return np.maximum(z, 0.0)

    return T


def estimate_alignment_gap(real_scores, reference_scores, T, grid_size=400):
    """
    Diagnostic gap between T#P_real and P_ref using empirical quantiles.
    """

    transformed = T(real_scores)

    u = np.linspace(0.01, 0.99, grid_size)

    q_transformed = np.quantile(transformed, u)
    q_reference = np.quantile(reference_scores, u)

    return float(np.mean(np.abs(q_transformed - q_reference)))


# ============================================================
# Correct guarded RSA-CP inclusion rule
# ============================================================

def rsa_cp_include_scores_guarded(
    scores,
    real_scores,
    reference_scores,
    alpha,
    beta,
    T,
    precomputed=None,
):
    """
    Guarded RSA-CP inclusion rule.

    For each candidate/test score s:

        z = T(s)
        k = 1 + #{i: T(S_i) <= z}
        B = #{j: ref_j <= z}

    Include if either:

        lower guard:
            k + b_+(k,beta) <= j_star

    or:

        refined augmented-rank rule:
            B in [b_-(k,beta), b_+(k,beta)]
            and k + B <= j_star.

    The lower guard ensures that under severe mismatch RSA-CP reduces to
    the certified lower-bound rule instead of under-covering.
    """

    scores = np.asarray(scores, dtype=float)
    real_scores = np.asarray(real_scores, dtype=float)
    reference_scores = np.asarray(reference_scores, dtype=float)

    m = len(real_scores)
    N = len(reference_scores)

    if precomputed is None:
        j_star = math.ceil((1 - alpha) * (m + N + 1))
        bminus, bplus = beta_binomial_windows(m, N, beta)
    else:
        j_star, bminus, bplus = precomputed

    z_real = np.sort(T(real_scores))
    z_scores = T(scores)
    ref_sorted = np.sort(reference_scores)

    # Real-score rank contribution.
    k = 1 + np.searchsorted(z_real, z_scores, side="right")
    k = np.clip(k, 1, m + 1)

    # Observed reference-rank contribution.
    B = np.searchsorted(ref_sorted, z_scores, side="right")

    augmented_rank = k + B

    # Certified lower-bound guard.
    lower_guard_include = (k + bplus[k]) <= j_star

    # Beta-Binomial compatibility.
    compatible = (B >= bminus[k]) & (B <= bplus[k])

    # Refined augmented-rank inclusion.
    augmented_include = augmented_rank <= j_star

    # Final RSA-CP rule.
    include = lower_guard_include | (compatible & augmented_include)

    diagnostics = {
        "k": k,
        "B": B,
        "augmented_rank": augmented_rank,
        "j_star": j_star,
        "bminus": bminus[k],
        "bplus": bplus[k],
        "compatible": compatible,
        "lower_guard_include": lower_guard_include,
        "augmented_include": augmented_include,
        "include": include,
    }

    return include.astype(bool), diagnostics


# ============================================================
# Width estimation
# ============================================================

def estimate_score_set_width(
    real_scores,
    reference_scores,
    alpha,
    beta,
    T,
    precomputed,
    grid_size=5000,
):
    """
    Estimate prediction-set width for absolute-error scores.

    Since inclusion is candidate-wise, we estimate the measure of the
    accepted nonnegative score-radius set on a grid and multiply by 2.
    """

    upper = max(
        np.max(real_scores),
        np.max(reference_scores),
        1.0,
    )

    accepted = None
    grid = None

    for _ in range(8):
        grid = np.linspace(0.0, upper, grid_size)

        accepted, _ = rsa_cp_include_scores_guarded(
            scores=grid,
            real_scores=real_scores,
            reference_scores=reference_scores,
            alpha=alpha,
            beta=beta,
            T=T,
            precomputed=precomputed,
        )

        if not accepted[-1]:
            break

        upper *= 2.0

    score_measure = np.trapz(accepted.astype(float), grid)
    width = 2.0 * score_measure

    max_radius = float(np.max(grid[accepted])) if np.any(accepted) else 0.0

    return float(width), max_radius


# ============================================================
# One trial
# ============================================================

def run_one_trial(rng, setting, precomputed):
    """
    Real, reference, and test scores are generated from the same distribution.

    The only thing that changes over the grid is the alignment map T_delta.
    """

    real_scores = sample_scores(rng, M, setting)
    reference_scores = sample_scores(rng, N_REF, setting)
    test_scores = sample_scores(rng, N_TEST, setting)

    rows = []

    # SCP baseline.
    q_scp = scp_quantile(real_scores, ALPHA)
    scp_included = scp_include_scores(test_scores, real_scores, ALPHA)

    cov_scp = float(np.mean(scp_included))
    width_scp = np.inf if np.isinf(q_scp) else float(2.0 * q_scp)

    for delta in ALIGNMENT_DRIFT_GRID:
        T_delta = make_alignment_map(
            delta=delta,
            real_scores=real_scores,
        )

        include_rsa, diag_rsa = rsa_cp_include_scores_guarded(
            scores=test_scores,
            real_scores=real_scores,
            reference_scores=reference_scores,
            alpha=ALPHA,
            beta=BETA,
            T=T_delta,
            precomputed=precomputed,
        )

        width_rsa, radius_rsa = estimate_score_set_width(
            real_scores=real_scores,
            reference_scores=reference_scores,
            alpha=ALPHA,
            beta=BETA,
            T=T_delta,
            precomputed=precomputed,
            grid_size=5000,
        )

        align_gap = estimate_alignment_gap(
            real_scores=real_scores,
            reference_scores=reference_scores,
            T=T_delta,
        )

        rows.append({
            "Setting": setting,
            "Alignment_Drift": delta,
            "Method": "SCP",
            "Coverage": cov_scp,
            "Width": width_scp,
            "Max_Radius": q_scp,
            "Compat_Rate": np.nan,
            "Lower_Guard_Rate": np.nan,
            "Augmented_Use_Rate": np.nan,
            "Include_Rate": np.nan,
            "Mean_B": np.nan,
            "Mean_K": np.nan,
            "Mean_Augmented_Rank": np.nan,
            "Alignment_Gap": align_gap,
        })

        rows.append({
            "Setting": setting,
            "Alignment_Drift": delta,
            "Method": "RSA-CP",
            "Coverage": float(np.mean(include_rsa)),
            "Width": width_rsa,
            "Max_Radius": radius_rsa,
            "Compat_Rate": float(np.mean(diag_rsa["compatible"])),
            "Lower_Guard_Rate": float(np.mean(diag_rsa["lower_guard_include"])),
            "Augmented_Use_Rate": float(
                np.mean(diag_rsa["compatible"] & diag_rsa["augmented_include"])
            ),
            "Include_Rate": float(np.mean(diag_rsa["include"])),
            "Mean_B": float(np.mean(diag_rsa["B"])),
            "Mean_K": float(np.mean(diag_rsa["k"])),
            "Mean_Augmented_Rank": float(np.mean(diag_rsa["augmented_rank"])),
            "Alignment_Gap": align_gap,
        })

    return rows


# ============================================================
# Full experiment
# ============================================================

def run_experiment():
    rng = np.random.default_rng(SEED)

    bminus, bplus = beta_binomial_windows(M, N_REF, BETA)

    j_star = math.ceil((1 - ALPHA) * (M + N_REF + 1))

    precomputed = (j_star, bminus, bplus)

    prop_lower, prop_upper = prop33_rank_bounds(
        M,
        N_REF,
        ALPHA,
        BETA,
    )

    rows = []

    for trial in range(1, N_TRIALS + 1):
        if trial % 25 == 0:
            print(f"Trial {trial}/{N_TRIALS}")

        for setting in SETTINGS:
            trial_rows = run_one_trial(
                rng=rng,
                setting=setting,
                precomputed=precomputed,
            )

            for row in trial_rows:
                row["Trial"] = trial
                row["Target"] = 1 - ALPHA
                row["Prop33_Lower"] = prop_lower
                row["Prop33_Upper"] = prop_upper
                row["j_star"] = j_star
                rows.append(row)

    raw = pd.DataFrame(rows)

    def finite_mean(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return float(np.mean(x)) if len(x) else np.inf

    def finite_se(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        if len(x) <= 1:
            return np.nan
        return float(np.std(x, ddof=1) / np.sqrt(len(x)))

    summary = (
        raw.groupby(["Setting", "Alignment_Drift", "Method"], as_index=False)
        .agg(
            Mean_Coverage=("Coverage", "mean"),
            SE_Coverage=(
                "Coverage",
                lambda x: np.std(x, ddof=1) / np.sqrt(len(x)),
            ),
            Mean_Width=("Width", finite_mean),
            SE_Width=("Width", finite_se),
            Mean_Max_Radius=("Max_Radius", finite_mean),
            SE_Max_Radius=("Max_Radius", finite_se),
            Mean_Compat_Rate=("Compat_Rate", "mean"),
            Mean_Lower_Guard_Rate=("Lower_Guard_Rate", "mean"),
            Mean_Augmented_Use_Rate=("Augmented_Use_Rate", "mean"),
            Mean_Include_Rate=("Include_Rate", "mean"),
            Mean_B=("Mean_B", "mean"),
            Mean_K=("Mean_K", "mean"),
            Mean_Augmented_Rank=("Mean_Augmented_Rank", "mean"),
            Mean_Alignment_Gap=("Alignment_Gap", "mean"),
            Target=("Target", "first"),
            Prop33_Lower=("Prop33_Lower", "first"),
            Prop33_Upper=("Prop33_Upper", "first"),
            j_star=("j_star", "first"),
            n_runs=("Coverage", "count"),
        )
    )

    return raw, summary


# ============================================================
# Plotting
# ============================================================

def plot_width(summary):
    for setting in SETTINGS:
        sub = summary[summary["Setting"] == setting]

        plt.figure(figsize=(7, 4.5))

        for method in ["SCP", "RSA-CP"]:
            s = sub[sub["Method"] == method].sort_values("Alignment_Drift")

            plt.errorbar(
                s["Alignment_Drift"],
                s["Mean_Width"],
                yerr=1.96 * s["SE_Width"],
                marker="o",
                capsize=3,
                linewidth=2,
                label=method,
            )

        plt.xlabel("Alignment-map drift")
        plt.ylabel("Average prediction-set width")
        plt.title(f"Width under Alignment-Map Drift ({setting})")
        plt.legend()
        plt.tight_layout()

        plt.savefig(OUT_DIR / f"width_alignment_drift_{setting}.png", dpi=300)
        plt.close()


def plot_coverage(summary):
    for setting in SETTINGS:
        sub = summary[summary["Setting"] == setting]

        plt.figure(figsize=(7, 4.5))

        for method in ["SCP", "RSA-CP"]:
            s = sub[sub["Method"] == method].sort_values("Alignment_Drift")

            plt.errorbar(
                s["Alignment_Drift"],
                s["Mean_Coverage"],
                yerr=1.96 * s["SE_Coverage"],
                marker="o",
                capsize=3,
                linewidth=2,
                label=method,
            )

        plt.axhline(
            1 - ALPHA,
            linestyle="--",
            linewidth=2,
            label="Target",
        )

        plt.axhline(
            sub["Prop33_Lower"].iloc[0],
            linestyle=":",
            linewidth=2,
            label="Prop. 3.3 lower bound",
        )

        #plt.xlabel("Alignment-map drift (\delta)")
        plt.xlabel(r"Alignment-map drift ($\delta$)")
        plt.ylabel("Empirical coverage")
        #plt.title(f"Coverage under Alignment-Map Drift ({setting})")
        plt.ylim(0.85, 1.02)
        plt.legend()
        plt.tight_layout()

        plt.savefig(OUT_DIR / f"coverage_alignment_drift_{setting}.png", dpi=300)
        plt.close()


def plot_diagnostics(summary):
    diag_cols = [
        ("Mean_Compat_Rate", "Compatibility rate"),
        ("Mean_Lower_Guard_Rate", "Lower-guard inclusion rate"),
        ("Mean_Augmented_Use_Rate", "Augmented-use rate"),
        ("Mean_Include_Rate", "Overall include rate"),
        ("Mean_B", "Mean reference rank contribution B"),
        ("Mean_Augmented_Rank", "Mean augmented rank k+B"),
        ("Mean_Alignment_Gap", "Alignment gap"),
    ]

    for setting in SETTINGS:
        rsa = summary[
            (summary["Setting"] == setting)
            & (summary["Method"] == "RSA-CP")
        ]

        for col, label in diag_cols:
            plt.figure(figsize=(7, 4.5))

            s = rsa.sort_values("Alignment_Drift")

            plt.plot(
                s["Alignment_Drift"],
                s[col],
                marker="o",
                linewidth=2,
            )

            plt.xlabel("Alignment-map drift")
            plt.ylabel(label)
            plt.title(f"{label} ({setting})")
            plt.tight_layout()

            safe_col = col.lower().replace("mean_", "").replace("_", "-")
            plt.savefig(OUT_DIR / f"{safe_col}_{setting}.png", dpi=300)
            plt.close()


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    raw, summary = run_experiment()

    raw.to_csv(
        OUT_DIR / "raw_alignment_drift_guarded_results.csv",
        index=False,
    )

    summary.to_csv(
        OUT_DIR / "summary_alignment_drift_guarded_results.csv",
        index=False,
    )

    print(summary.to_string(index=False))

    plot_width(summary)
    plot_coverage(summary)
    plot_diagnostics(summary)

    print(f"\nSaved results to: {OUT_DIR.resolve()}")